# resist. — YOLOv8 + Color Classifier Training

**What this notebook does:**
1. Trains a YOLOv8n model to detect individual resistor bands (bounding boxes)
2. Builds a synthetic color patch dataset for the 12 resistor colors
3. Trains a small CNN to classify each detected band's color
4. Exports both models to ONNX for zero-dependency inference
5. Downloads both `.onnx` files to your Google Drive

**Before you start:**
- Runtime → Change runtime type → **T4 GPU**
- You need a free [Roboflow](https://roboflow.com) account to download the dataset
- Estimated time: ~25 minutes end-to-end

---

## Step 0 — Install packages and mount Drive

In [ ]:
# Install everything we need
# ultralytics = YOLOv8 framework
# roboflow    = dataset download
# onnx + onnxruntime = export + local inference

!pip install -q ultralytics roboflow onnx onnxruntime
print('✅ Packages installed')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# All outputs (trained models, exports) go here
import os
SAVE_DIR = '/content/drive/MyDrive/resist_models'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ Drive mounted. Models will be saved to: {SAVE_DIR}')

In [ ]:
# Verify GPU is available
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
else:
    print('⚠️  No GPU found. Make sure you set Runtime → Change runtime type → T4 GPU')

---
## Step 1 — Download the resistor dataset from Roboflow

**How to get your API key:**
1. Go to [roboflow.com](https://roboflow.com) and create a free account
2. Click your profile icon (top right) → **Settings** → **Roboflow API**
3. Copy the **Private API Key** and paste it below

We're using the **"Resistor-Band-Detection"** dataset which has images annotated with
individual band bounding boxes labeled with their color class.

In [ ]:
# ⚠️ PASTE YOUR ROBOFLOW API KEY HERE
ROBOFLOW_API_KEY = 'YOUR_KEY_HERE'

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Primary dataset: Resistor band detection with color labels
# Source: https://universe.roboflow.com/resistor-jcqbr/resistor-band-detection
project = rf.workspace('resistor-jcqbr').project('resistor-band-detection')
dataset = project.version(1).download('yolov8')

DATASET_PATH = dataset.location
print(f'✅ Dataset downloaded to: {DATASET_PATH}')

In [ ]:
# Inspect what we downloaded
import os, yaml

data_yaml = os.path.join(DATASET_PATH, 'data.yaml')
with open(data_yaml) as f:
    cfg = yaml.safe_load(f)

print('Classes:', cfg.get('names'))
print('Number of classes:', cfg.get('nc'))
print('Train images:', len(os.listdir(os.path.join(DATASET_PATH, 'train', 'images'))))
print('Val images:', len(os.listdir(os.path.join(DATASET_PATH, 'valid', 'images'))))

---
## Step 2 — Train YOLOv8n Band Detector

We're training `yolov8n` (nano — the smallest, fastest variant).
It's enough for detecting colored bands, and it'll fit inside your Flask app without weighing 200MB.

**Key parameters explained:**
- `epochs=80` — how many full passes over the training data. More = better up to a point.
- `imgsz=640` — input image size. Standard for YOLOv8.
- `batch=16` — images processed at once. 16 fits comfortably on a T4.
- `patience=20` — stop early if no improvement for 20 epochs (saves time).


In [ ]:
from ultralytics import YOLO

# Load the pretrained nano model (downloads ~6MB weights automatically)
yolo_model = YOLO('yolov8n.pt')

# Train!
results = yolo_model.train(
    data=data_yaml,
    epochs=80,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,           # GPU 0
    project='/content/runs',
    name='band_detector',
    exist_ok=True,
    verbose=True,
)

YOLO_BEST = '/content/runs/band_detector/weights/best.pt'
print(f'✅ Training done. Best weights: {YOLO_BEST}')

In [ ]:
# Quick validation — shows mAP50 and mAP50-95
# mAP50 > 0.70 = good enough to deploy
# mAP50 > 0.85 = very good

val_model = YOLO(YOLO_BEST)
metrics = val_model.val(data=data_yaml, device=0)
print(f'mAP50: {metrics.box.map50:.3f}')
print(f'mAP50-95: {metrics.box.map:.3f}')

if metrics.box.map50 < 0.65:
    print('⚠️  mAP50 is below 0.65 — consider running more epochs or augmenting data')
else:
    print('✅ Accuracy looks good!')

In [ ]:
# Visualise some predictions so you can eyeball accuracy
import glob
from IPython.display import Image, display

val_imgs = glob.glob(os.path.join(DATASET_PATH, 'valid', 'images', '*.jpg'))[:4]
pred_model = YOLO(YOLO_BEST)

for img_path in val_imgs:
    result = pred_model(img_path, save=True, project='/content/preview', name='val', exist_ok=True)

for img in glob.glob('/content/preview/val/*.jpg')[:4]:
    display(Image(filename=img, width=500))

---
## Step 3 — Build a synthetic color patch dataset

The Roboflow dataset gives us band *locations* but not always reliable color labels at pixel level.
For the color classifier we generate synthetic training data:
- 2000 patches per color class with realistic HSV variation
- Mimics how a real resistor band looks under different lighting

This approach works extremely well for solid-color classification.

In [ ]:
import numpy as np
import cv2
import os
from pathlib import Path

# 12 resistor colors: name → (H_center, H_range, S_min, S_max, V_min, V_max)
# H in OpenCV is 0–179, S and V are 0–255
COLOR_SPECS = {
    'black':  (0,   0,   0,   30,  0,   50),
    'brown':  (10,  5,   80,  180, 40,  100),
    'red':    (0,   5,   150, 255, 100, 200),
    'orange': (12,  4,   180, 255, 140, 230),
    'yellow': (28,  5,   180, 255, 180, 255),
    'green':  (55,  10,  80,  200, 40,  140),
    'blue':   (105, 10,  100, 220, 60,  180),
    'violet': (130, 10,  80,  200, 50,  160),
    'gray':   (0,   0,   0,   30,  80,  180),
    'white':  (0,   0,   0,   20,  200, 255),
    'gold':   (22,  5,   80,  160, 130, 200),
    'silver': (0,   0,   0,   25,  150, 210),
}

SYNTH_DIR = '/content/color_dataset'
PATCH_SIZE = 64
SAMPLES_PER_CLASS = 2000

def make_patch(h_center, h_range, s_min, s_max, v_min, v_max):
    """Generate a 64x64 solid color patch with random HSV variation."""
    h = int(np.clip(np.random.normal(h_center, h_range / 2), 0, 179))
    s = int(np.random.uniform(s_min, s_max))
    v = int(np.random.uniform(v_min, v_max))

    # Add subtle noise to simulate real resistor surface texture
    patch_hsv = np.full((PATCH_SIZE, PATCH_SIZE, 3), [h, s, v], dtype=np.uint8)
    noise = np.random.randint(-8, 8, patch_hsv.shape, dtype=np.int16)
    patch_hsv = np.clip(patch_hsv.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    # Add a subtle gradient to simulate lighting variation
    gradient = np.linspace(0.85, 1.0, PATCH_SIZE).reshape(1, -1, 1)
    patch_hsv[:, :, 2] = np.clip(patch_hsv[:, :, 2] * gradient, 0, 255).astype(np.uint8)

    return cv2.cvtColor(patch_hsv, cv2.COLOR_HSV2BGR)

print('Generating synthetic color patches...')

class_names = list(COLOR_SPECS.keys())

for split in ['train', 'val']:
    n = SAMPLES_PER_CLASS if split == 'train' else 300
    for color, spec in COLOR_SPECS.items():
        out_dir = os.path.join(SYNTH_DIR, split, color)
        os.makedirs(out_dir, exist_ok=True)
        for i in range(n):
            patch = make_patch(*spec)
            cv2.imwrite(os.path.join(out_dir, f'{i:04d}.jpg'), patch)

print(f'✅ Dataset created: {SAMPLES_PER_CLASS} train + 300 val patches per class ({len(class_names)} classes)')
print('Classes:', class_names)

In [ ]:
# Visualise one sample from each color class
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
axes = axes.flatten()

for i, color in enumerate(class_names):
    sample_path = os.path.join(SYNTH_DIR, 'train', color, '0000.jpg')
    img = cv2.imread(sample_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img_rgb)
    axes[i].set_title(color, fontsize=10)
    axes[i].axis('off')

plt.suptitle('Synthetic color patches (one sample each)', y=1.02)
plt.tight_layout()
plt.show()

---
## Step 4 — Train the CNN Color Classifier

A simple 4-layer CNN. Input: 64×64 RGB patch. Output: one of 12 color classes.
Trains in ~5 minutes on T4.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import CosineAnnealingLR

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {DEVICE}')

# ── Transforms ──────────────────────────────────────────────────────────────
# Training: augment to help the model generalise to real resistor photos
train_tf = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

val_tf = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

train_ds = datasets.ImageFolder(os.path.join(SYNTH_DIR, 'train'), transform=train_tf)
val_ds   = datasets.ImageFolder(os.path.join(SYNTH_DIR, 'val'),   transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False, num_workers=2)

# The class order from ImageFolder (alphabetical) — we'll save this for inference
CLASS_NAMES = train_ds.classes
print('Class order (IMPORTANT — save this):', CLASS_NAMES)

NUM_CLASSES = len(CLASS_NAMES)

In [ ]:
class ColorCNN(nn.Module):
    """
    Small CNN: 4 conv blocks + classifier head.
    ~400k parameters — fast to train, fast to run on CPU.
    Input:  (B, 3, 64, 64) — BGR image patch, normalised to [-1, 1]
    Output: (B, 12) — logits for 12 color classes
    """
    def __init__(self, num_classes: int = 12):
        super().__init__()

        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )

        self.features = nn.Sequential(
            conv_block(3,  32),   # 64 → 32
            conv_block(32, 64),   # 32 → 16
            conv_block(64, 128),  # 16 → 8
            conv_block(128, 256), # 8  → 4
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = ColorCNN(num_classes=NUM_CLASSES).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')

In [ ]:
EPOCHS = 40
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

CNN_BEST = '/content/color_cnn_best.pt'
best_val_acc = 0.0

print(f'Training for {EPOCHS} epochs...\n')

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    train_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # ── Validate ──
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            preds = model(imgs).argmax(1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    val_acc = correct / total
    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'model_state': model.state_dict(),
                    'class_names': CLASS_NAMES,
                    'num_classes': NUM_CLASSES}, CNN_BEST)

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | '
              f'Loss: {train_loss/len(train_loader):.4f} | '
              f'Val acc: {val_acc*100:.1f}% | '
              f'Best: {best_val_acc*100:.1f}%')

print(f'\n✅ Training complete. Best validation accuracy: {best_val_acc*100:.1f}%')

if best_val_acc < 0.90:
    print('⚠️  Accuracy below 90%. Consider adjusting HSV ranges in COLOR_SPECS above')
else:
    print('🎉 Accuracy ≥ 90% — ready to export!')

In [ ]:
# Confusion matrix on validation set so you can see which colors get mixed up
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

checkpoint = torch.load(CNN_BEST, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state'])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, colorbar=False, xticks_rotation=45)
plt.title('Color Classifier — Confusion Matrix')
plt.tight_layout()
plt.show()

---
## Step 5 — Export both models to ONNX

ONNX (Open Neural Network Exchange) lets you run the models on any platform
**without** needing PyTorch or CUDA installed. Just `onnxruntime` (~10MB pip package).
Perfect for running inside Flask on a cheap AWS instance.

In [ ]:
import onnx
import onnxruntime as ort

# ── Export CNN Color Classifier ──────────────────────────────────────────────
CNN_ONNX = '/content/color_classifier.onnx'

checkpoint = torch.load(CNN_BEST, map_location='cpu')
model_cpu = ColorCNN(num_classes=NUM_CLASSES)
model_cpu.load_state_dict(checkpoint['model_state'])
model_cpu.eval()

dummy_input = torch.randn(1, 3, 64, 64)

torch.onnx.export(
    model_cpu,
    dummy_input,
    CNN_ONNX,
    input_names=['image'],
    output_names=['logits'],
    dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
)

# Verify ONNX export
onnx_model = onnx.load(CNN_ONNX)
onnx.checker.check_model(onnx_model)

# Quick runtime test
sess = ort.InferenceSession(CNN_ONNX, providers=['CPUExecutionProvider'])
test_out = sess.run(['logits'], {'image': dummy_input.numpy()})
print(f'✅ CNN exported to ONNX. Output shape: {test_out[0].shape}')
print(f'   File size: {os.path.getsize(CNN_ONNX) / 1024:.0f} KB')

In [ ]:
# ── Export YOLOv8 Band Detector ──────────────────────────────────────────────
yolo_export_model = YOLO(YOLO_BEST)
yolo_export_model.export(format='onnx', dynamic=True, simplify=True)

# The exported file lands next to the .pt file
YOLO_ONNX_SRC = YOLO_BEST.replace('.pt', '.onnx')
YOLO_ONNX = '/content/band_detector.onnx'

import shutil
shutil.copy(YOLO_ONNX_SRC, YOLO_ONNX)

print(f'✅ YOLO exported to ONNX')
print(f'   File size: {os.path.getsize(YOLO_ONNX) / 1024 / 1024:.1f} MB')

In [ ]:
# ── Save class names alongside models ──────────────────────────────────────
import json

# Save color classifier class list (order matters — must match training)
CLASSES_JSON = '/content/color_classes.json'
with open(CLASSES_JSON, 'w') as f:
    json.dump(CLASS_NAMES, f)
print('Color classes:', CLASS_NAMES)

# Save YOLO class list
YOLO_CLASSES_JSON = '/content/yolo_classes.json'
with open(data_yaml) as f:
    yolo_cfg = yaml.safe_load(f)
with open(YOLO_CLASSES_JSON, 'w') as f:
    json.dump(yolo_cfg.get('names', []), f)

print('YOLO classes:', yolo_cfg.get('names'))

In [ ]:
# ── Copy everything to Google Drive ─────────────────────────────────────────
files_to_save = [
    (YOLO_ONNX,       'band_detector.onnx'),
    (CNN_ONNX,        'color_classifier.onnx'),
    (CLASSES_JSON,    'color_classes.json'),
    (YOLO_CLASSES_JSON, 'yolo_classes.json'),
]

for src, fname in files_to_save:
    dst = os.path.join(SAVE_DIR, fname)
    shutil.copy(src, dst)
    size_kb = os.path.getsize(dst) / 1024
    print(f'✅ Saved {fname} → Drive ({size_kb:.0f} KB)')

print(f'\n🎉 All done! Your models are in Google Drive at:\n{SAVE_DIR}')
print('\nNext: download these 4 files into your project at inference/models/')

---
## Step 6 — Smoke test (run inference on a sample image)

Let's verify the full pipeline works end-to-end before you download anything.

In [ ]:
import cv2
import numpy as np
import onnxruntime as ort
import json
from IPython.display import Image, display

# Load models
yolo_sess   = ort.InferenceSession(YOLO_ONNX,  providers=['CPUExecutionProvider'])
color_sess  = ort.InferenceSession(CNN_ONNX,   providers=['CPUExecutionProvider'])

with open(CLASSES_JSON) as f:
    color_classes = json.load(f)

def preprocess_yolo(img_bgr, size=640):
    h, w = img_bgr.shape[:2]
    scale = size / max(h, w)
    nh, nw = int(h * scale), int(w * scale)
    resized = cv2.resize(img_bgr, (nw, nh))
    pad = np.full((size, size, 3), 114, dtype=np.uint8)
    pad[:nh, :nw] = resized
    x = pad[:, :, ::-1].astype(np.float32) / 255.0
    return np.transpose(x, (2, 0, 1))[None], scale, (h, w)

def classify_color(crop_bgr):
    crop = cv2.resize(crop_bgr, (64, 64))
    x = (crop[:, :, ::-1].astype(np.float32) / 127.5) - 1.0  # normalise to [-1,1]
    x = np.transpose(x, (2, 0, 1))[None]
    logits = color_sess.run(['logits'], {'image': x})[0][0]
    idx = int(np.argmax(logits))
    conf = float(np.exp(logits[idx]) / np.sum(np.exp(logits)))
    return color_classes[idx], conf

# Pick a val image to test on
test_imgs = glob.glob(os.path.join(DATASET_PATH, 'valid', 'images', '*.jpg'))
if not test_imgs:
    test_imgs = glob.glob(os.path.join(DATASET_PATH, 'valid', 'images', '*.png'))

img_path = test_imgs[0]
img = cv2.imread(img_path)

inp, scale, (orig_h, orig_w) = preprocess_yolo(img)
outputs = yolo_sess.run(None, {yolo_sess.get_inputs()[0].name: inp})[0][0]

# Parse detections: [x1,y1,x2,y2,conf,cls] format for YOLOv8 ONNX
# YOLOv8 exports as (8400, num_classes+4) — need to transpose + filter
preds = outputs.T  # (8400, 4+nc)
scores = preds[:, 4:].max(axis=1)
mask = scores > 0.35
boxes_raw = preds[mask, :4]   # cx, cy, w, h (normalised to 640)
confs = scores[mask]
classes = preds[mask, 4:].argmax(axis=1)

print(f'Detected {mask.sum()} bands\n')
result_img = img.copy()

for box, conf, cls in zip(boxes_raw, confs, classes):
    cx, cy, bw, bh = box / scale
    x1 = int(cx - bw/2); y1 = int(cy - bh/2)
    x2 = int(cx + bw/2); y2 = int(cy + bh/2)
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(orig_w, x2), min(orig_h, y2)

    crop = img[y1:y2, x1:x2]
    if crop.size == 0:
        continue

    color_name, color_conf = classify_color(crop)
    label = f'{color_name} ({color_conf:.0%})'
    print(f'  Band detected — color: {color_name} confidence: {color_conf:.1%}')
    cv2.rectangle(result_img, (x1, y1), (x2, y2), (0, 200, 255), 2)
    cv2.putText(result_img, label, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0,200,255), 1)

out_path = '/content/test_result.jpg'
cv2.imwrite(out_path, result_img)
display(Image(filename=out_path, width=600))
print('\n✅ Pipeline working. You can now download your models from Google Drive.')

---
## Done! What to do next

1. Download these 4 files from `Google Drive → resist_models/` to your project:
   - `band_detector.onnx` — band locator
   - `color_classifier.onnx` — color reader
   - `color_classes.json` — class name list
   - `yolo_classes.json` — YOLO class list

2. Place them inside your project at:
   ```
   resist/
   └── inference/
       └── models/
           ├── band_detector.onnx
           ├── color_classifier.onnx
           ├── color_classes.json
           └── yolo_classes.json
   ```

3. In your Flask app, replace the Gemini `/analyze` route with the local detector.
   The `inference/detector.py` file in your repo handles this automatically.

4. `requirements.txt` change: replace `google-generativeai` with `onnxruntime`

---
*If the confusion matrix shows black/brown being confused: try narrowing their V ranges.*  
*If gold/silver are confused: gold has hue ~22, silver has no saturation — check your COLOR_SPECS.*